Dataset:
https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

# SETUP FOR SQL



In [ ]:
# 1. Install the modern SQL tool
!pip install jupysql --quiet

# 2. Load the extension so Colab understands %sql and %%sql
%load_ext sql

# 3. Set it to output clean Pandas tables
%config SqlMagic.autopandas = True

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [ ]:
import pandas as pd
import sqlite3

# 1. Create and connect to the newly named project database
conn = sqlite3.connect('olist_ecommerce.db')

# 2. List of all your CSV filenames (without the .csv extension)
datasets = [
    'orders_dataset',
    'product_category_name_translation',
    'products_dataset',
    'sellers_dataset',
    'customers_dataset',
    'geolocation_dataset',
    'order_items_dataset',
    'order_payments_dataset',
    'order_reviews_dataset'
]

# 3. Loop through the list, read the CSV, and load it into the database
for name in datasets:
    # Read the CSV file into a Pandas DataFrame
    df = pd.read_csv(f'{name}.csv')

    # Write the data to a SQLite table named after the file
    df.to_sql(name, conn, if_exists='replace', index=False)
    print(f"Successfully loaded: {name}")

print("\nAll 9 datasets are ready for querying!")

Successfully loaded: orders_dataset
Successfully loaded: product_category_name_translation
Successfully loaded: products_dataset
Successfully loaded: sellers_dataset
Successfully loaded: customers_dataset
Successfully loaded: geolocation_dataset
Successfully loaded: order_items_dataset
Successfully loaded: order_payments_dataset
Successfully loaded: order_reviews_dataset

All 9 datasets are ready for querying!


In [ ]:
%sql sqlite:///olist_ecommerce.db

# TIER 1: Data Exploration with  basics

The Tier 1 Foundation Checklist

* Core Filtering: WHERE, AND, OR, IN, BETWEEN

* Pattern Matching: LIKE (using % and _ wildcards)

* Null Handling: IS NULL, IS NOT NULL

* Sorting & Slicing: ORDER BY (ASC/DESC), LIMIT

* Aggregations: COUNT(), SUM(), AVG(), MIN(), MAX()

* Grouping: Basic GROUP BY

Master Query 1: The Logistics Audit

Target Table: orders_dataset

Concepts Tested: WHERE, AND, OR, IS NOT NULL, ORDER BY, LIMIT

The Business Question: "Show me the 10 most recent orders that are either 'shipped' or 'delivered', but only include the ones where the delivery date to the customer is actually recorded (not missing). Sort them by the purchase timestamp from newest to oldest."

In [ ]:
%%sql
SELECT
    order_id,
    order_status,
    order_purchase_timestamp,
    order_delivered_customer_date
FROM orders_dataset
WHERE (order_status = 'shipped' OR order_status = 'delivered')
  AND order_delivered_customer_date IS NOT NULL
ORDER BY order_purchase_timestamp DESC
LIMIT 10;

Running query in 'sqlite:///olist_ecommerce.db'

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date
0,35a972d7f8436f405b56e36add1a7140,delivered,2018-08-29 15:00:37,2018-08-30 16:23:36
1,03ef5dedbe7492bdae72eec50764c43f,delivered,2018-08-29 14:52:00,2018-08-30 16:36:59
2,168626408cb32af0ffaf76711caae1dc,delivered,2018-08-29 14:18:28,2018-08-30 16:52:31
3,0b223d92c27432930dfe407c6aea3041,delivered,2018-08-29 14:18:23,2018-08-30 16:24:55
4,52018484704db3661b98ce838612b507,delivered,2018-08-29 12:25:59,2018-08-30 22:48:27
5,d03ca98f59480e7e76c71fa83ecd8fb6,delivered,2018-08-29 11:06:11,2018-08-30 23:56:54
6,d70442bc5e3cb7438da497cc6a210f80,delivered,2018-08-29 10:22:35,2018-08-30 16:03:19
7,912859fef5a0bd5059b6d48fa79d121a,delivered,2018-08-29 09:48:09,2018-08-30 23:28:52
8,fb393211459aac00af932cd7ab4fa2cc,delivered,2018-08-29 09:14:11,2018-08-30 13:03:28
9,bee12e8653a04e76786e8891cfb6330a,delivered,2018-08-29 08:46:11,2018-08-30 21:54:45


Master Query 2: The Geo-Targeting Filter

Target Table: customers_dataset

Concepts Tested: WHERE, IN, LIKE (%), AND, OR

The Business Question: "Marketing wants to run a localized campaign. Find all customers located in the states of São Paulo ('SP'), Rio de Janeiro ('RJ'), or Minas Gerais ('MG'), but ONLY if their city name starts with 'sao' or contains the word 'rio' anywhere in it."

In [ ]:
%%sql
SELECT CUSTOMER_ID, CUSTOMER_CITY, CUSTOMER_STATE
FROM customers_dataset
WHERE CUSTOMER_STATE IN ('SP','RJ','MG')
AND CUSTOMER_CITY LIKE 'SAO%' OR '%RIO%';

Running query in 'sqlite:///olist_ecommerce.db'

,customer_id,customer_city,customer_state
0,18955e83d337fd6b2def6b18a428ac77,sao bernardo do campo,SP
1,4e7b3e00288586ebd08712fdd0374a03,sao paulo,SP
2,fd826e7cf63160e536e0908c76c3f441,sao paulo,SP
3,eabebad39a88bb6f5b52376faec28612,sao paulo,SP
4,9b8ce803689b3562defaad4613ef426f,sao paulo,SP
...,...,...,...
19552,da37711b17efd5f2539e8196ab215f04,sao paulo,SP
19553,821a7275a08f32975caceff2e08ea262,sao paulo,SP
19554,c6ece8a5137f3c9c3a3a12302a19a2ac,sao paulo,SP
19555,935993f47af1ed7d0715c26b686341c5,sao jose dos campos,SP


Master Query 3: The Revenue Pulse

Target Table: order_payments_dataset

Concepts Tested: COUNT(), SUM(), AVG(), MAX(), GROUP BY, ORDER BY, LIMIT

The Business Question: "What are the top 3 payment methods that generated the highest total revenue?
For each method, I also want to see the
* total number of transactions,
* the average ticket size,
* the single largest payment ever made using that method."

In [ ]:
%%sql
SELECT
    PAYMENT_TYPE,
    COUNT(ORDER_ID) AS TOTAL_TRANSACTIONS,
    SUM(PAYMENT_VALUE) AS TOTAL_REVENUE,
    AVG(PAYMENT_VALUE) AS AVG_TICKET_SIZE,
    MAX(PAYMENT_VALUE) AS MAX_PAYMENT
FROM order_payments_dataset
WHERE PAYMENT_TYPE != 'not_defined'
GROUP BY PAYMENT_TYPE
ORDER BY TOTAL_REVENUE DESC
LIMIT 3;

Running query in 'sqlite:///olist_ecommerce.db'

,payment_type,TOTAL_TRANSACTIONS,TOTAL_REVENUE,AVG_TICKET_SIZE,MAX_PAYMENT
0,credit_card,76795,1.254208e+07,163.319021,13664.08
1,boleto,19784,2.869361e+06,145.034435,7274.88
2,voucher,5775,3.794369e+05,65.703354,3184.34


Master Query 4: The Inventory Check

Target Table: products_dataset

Concepts Tested: WHERE, BETWEEN, IS NOT NULL, MIN(), MAX(), COUNT(), GROUP BY

The Business Question: "We need to audit our mid-weight inventory.
1. Count the number of products in each category, but only look at items weighing between 1,000 grams (1kg) and 5,000 grams (5kg).
2. Exclude any products that don't have a category name assigned.
3. Show me the top 5 categories with the most products in this weight range."

In [ ]:
%%sql
SELECT
  product_category_name,
  COUNT(PRODUCT_ID) AS PRODUCT_COUNT,
  MIN(PRODUCT_WEIGHT_g) AS MIN_WEIGHT,
  MAX(PRODUCT_WEIGHT_g) AS MAX_WEIGHT
FROM products_dataset
WHERE PRODUCT_WEIGHT_g BETWEEN 1000 AND 5000
  AND product_category_name IS NOT NULL
GROUP BY product_category_name
ORDER BY PRODUCT_COUNT DESC
LIMIT 5 ;

Running query in 'sqlite:///olist_ecommerce.db'

,product_category_name,PRODUCT_COUNT,MIN_WEIGHT,MAX_WEIGHT
0,cama_mesa_banho,1553,1000.0,5000.0
1,moveis_decoracao,1051,1000.0,5000.0
2,esporte_lazer,957,1000.0,5000.0
3,utilidades_domesticas,792,1000.0,5000.0
4,beleza_saude,675,1000.0,4900.0


Master Query 5: The Seller Performance

Target Table: order_items_dataset

Concepts Tested: Combine almost everything (Math, Aggregations, Grouping, Filtering, Sorting)

The Business Question: "Evaluate our premium sellers during 2017.
1. Find the total revenue (price) and total freight costs for each seller,
2. but only for items priced strictly greater than $50,
3. and where the shipping limit date occurred anytime in the year 2017.
4. Show me the top 5 sellers by total revenue."

In [ ]:
%%sql
SELECT
    seller_id,
    COUNT(order_item_id) AS premium_items_sold,
    SUM(price) AS total_premium_revenue,
    SUM(freight_value) AS total_freight_costs
FROM order_items_dataset
WHERE price > 50
  AND shipping_limit_date BETWEEN '2017-01-01 00:00:00' AND '2017-12-31 23:59:59'
GROUP BY seller_id
ORDER BY total_premium_revenue DESC
LIMIT 5;

Running query in 'sqlite:///olist_ecommerce.db'

,seller_id,premium_items_sold,total_premium_revenue,total_freight_costs
0,53243585a1d6dc2643021fd1853d8905,255,176685.92,8062.74
1,7e93a43ef30c4f03f38b393420bc753a,283,159010.13,5353.53
2,4a3ca9315b744ce9f8e9374361493884,1130,122969.27,18799.02
3,46dc3b2cc0980fb8ec44634e21d2718e,451,108537.04,9618.33
4,fa1c13f2614d7b5c4749cbc52fecda94,280,96667.91,4473.76


# Tier 2: The Relational Sandbox (Medium).

The Tier 2 Master Checklist (Deduplicated Patterns)

* The Inner Join: Connecting fact tables to dimension tables.
* The Multi-Table Join: Chaining 3+ tables together.
* The Left Join (Orphan Hunt): Finding records in Table A that have no match in Table B.
* The HAVING Clause: Filtering data after it has been aggregated.
* The Scalar Subquery: Using a subquery in the WHERE clause to compare against a dynamically calculated value (like an average or max).
* The IN Subquery: Filtering based on a generated list.
* Date/String Aggregations: Grouping by extracted date parts (Month/Year).

## **Master Query 1: The Basic Bridge (INNER JOIN)**

Concepts Tested: INNER JOIN, table aliasing.

The Business Question: "We need a simple report connecting orders to their payment methods. Write a query that returns the order_id, order_status, payment_type, and payment_installments for the first 10 rows."

Tables Needed: orders_dataset, order_payments_dataset.

In [ ]:
%%sql
SELECT orders_dataset.order_id, order_status, payment_type, payment_installments
FROM orders_dataset
INNER JOIN order_payments_dataset
 ON orders_dataset.order_id = order_payments_dataset.order_id
LIMIT 10;
-- only ORDER_ID IS GIVING "AMBIGUOUS ERROR - WHY ? "

Running query in 'sqlite:///olist_ecommerce.db'

,order_id,order_status,payment_type,payment_installments
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,credit_card,1
1,e481f51cbdc54678b7cc49136f2d6af7,delivered,voucher,1
2,e481f51cbdc54678b7cc49136f2d6af7,delivered,voucher,1
3,53cdb2fc8bc7dce0b6741e2150273451,delivered,boleto,1
4,47770eb9100c2d0c44946d9cf07ec65d,delivered,credit_card,3
5,949d5b44dbf5de918fe9c16f97b45f8a,delivered,credit_card,1
6,ad21c59c0840e6cb83a9ceb5573f8159,delivered,credit_card,1
7,a4591c265e18cb1dcee52889e2d8acc3,delivered,credit_card,6
8,136cce7faa42fdb2cefd53fdc79a6098,invoiced,credit_card,1
9,6514b8ad8028c9f2cc2374ded245783f,delivered,credit_card,3


Your Logic: Your underlying logic is 100% correct. You correctly identified the
two tables, used an INNER JOIN (written simply as join, which is perfectly fine), and joined them on the right key (order_id). You also correctly used LIMIT 10.

The "Ambiguous Error" Explained:
You ran into the classic ambiguous column name: order_id error. Here is exactly why that happens:

Since you joined orders_dataset and order_payments_dataset, the SQL engine now has a wide, combined temporary table behind the scenes. Both of those original tables contain a column named order_id.

When you say SELECT order_id, the database throws its hands up and says, "Wait, do you want the order_id from the orders table, or the order_id from the payments table? They are identical, but I need you to be specific!" It doesn't throw this error for order_status or payment_type because those columns only exist in one of the tables, so there is no confusion.

The Fix & The "Better Way" (Table Aliasing):
To fix this, you just need to tell SQL which table to pull order_id from. While you could write orders_dataset.order_id in the SELECT clause, the industry standard (and much better way) is to use Table Aliases. This assigns a temporary nickname to your tables and saves you from typing long table names over and over.

In [ ]:
%%sql
SELECT
  o.order_id,
  o.order_status,
  op.payment_type,
  op.payment_installments
FROM orders_dataset AS o       -- 'o' is now the alias for the orders table
JOIN order_payments_dataset AS op  -- 'op' is now the alias for the payments table
  ON o.order_id = op.order_id
LIMIT 10;
-- (Note: The AS keyword for aliasing is optional, you can also just write orders_dataset o, but using AS makes it very readable when you are learning).

Running query in 'sqlite:///olist_ecommerce.db'

,order_id,order_status,payment_type,payment_installments
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,credit_card,1
1,e481f51cbdc54678b7cc49136f2d6af7,delivered,voucher,1
2,e481f51cbdc54678b7cc49136f2d6af7,delivered,voucher,1
3,53cdb2fc8bc7dce0b6741e2150273451,delivered,boleto,1
4,47770eb9100c2d0c44946d9cf07ec65d,delivered,credit_card,3
5,949d5b44dbf5de918fe9c16f97b45f8a,delivered,credit_card,1
6,ad21c59c0840e6cb83a9ceb5573f8159,delivered,credit_card,1
7,a4591c265e18cb1dcee52889e2d8acc3,delivered,credit_card,6
8,136cce7faa42fdb2cefd53fdc79a6098,invoiced,credit_card,1
9,6514b8ad8028c9f2cc2374ded245783f,delivered,credit_card,3


## **Master Query 2: The Supply Chain Trace (Multi-Table JOIN)**

Concepts Tested: Chaining multiple INNER JOINs.

The Business Question: "Let's track a product from the seller to the customer. Write a query that returns the customer_id, the customer_city, the product_id, and the seller_city for all delivered orders."

Tables Needed: orders_dataset, order_items_dataset, customers_dataset, sellers_dataset.

In [ ]:
%%sql
SELECT
  orders_dataset.customer_id,
  customer_city,
  product_id,
  seller_city
FROM orders_dataset
-- this is the main table which can inner join with all other 3 tables
INNER JOIN order_items_dataset
  ON orders_dataset.order_id = order_items_dataset.order_id
  INNER JOIN customers_dataset
  ON orders_dataset.customer_id = customers_dataset.customer_id
  INNER JOIN sellers_dataset
  ON order_items_dataset.seller_id = sellers_dataset.seller_id
  WHERE order_status = 'delivered'
  LIMIT 10;

Running query in 'sqlite:///olist_ecommerce.db'

,customer_id,customer_city,product_id,seller_city
0,9ef432eb6251297304e76186b10a928d,sao paulo,87285b34884572647811a353c7ac498a,maua
1,b0830fb4747a6c6d20dea0b8c802d7ef,barreiras,595fac2a385ac33a80bd5114aec74eb8,belo horizonte
2,41ce2a54c0b03bf3443c3d931a367089,vianopolis,aa4383b373c6aca5d8797843e5594415,guariba
3,f88197465ea7920adcdbec7375364d82,sao goncalo do amarante,d0b61bfb1de832b15ba9d266ca96e5b0,belo horizonte
4,8ab97904e6daea8866dbdbc4fb7aad2c,santo andre,65266b2da20d04dbe00c5c2d3bb7859e,mogi das cruzes
5,503740e9ca751ccdda7ba28e9ab8f608,congonhinhas,060cb19345d90064d1015407193c233d,guarulhos
6,9bdf08b4b3b52b5526ff42d37d47f222,nilopolis,4520766ec412348b8d4caa5e8a18c464,atibaia
7,f54a9f0e6b351c431402b8461ea51999,faxinalzinho,ac1789e492dcd698c5c10b97a671243a,sao jose do rio pardo
8,31ad1d1b63eb9962463f764d4e6e0c9d,sorocaba,9a78fb9862b10749a117f7fc3c31f051,itaquaquecetuba
9,494dded5b201313c64ed7f100595b95c,rio de janeiro,08574b074924071f4e201e151b152b4e,cariacica


Let's look at Master Query 2: The Supply Chain Trace.

This is a big one. Chaining four tables together is a major milestone because it proves you understand how relational databases are wired.

The Goal: Return customer_id, customer_city, product_id, and seller_city for all delivered orders.

The Logic & Review:
To get all this information, you have to walk the "bridge" from the customer all the way to the seller.

1. You start with orders_dataset because it's the central hub (and has the order_status we need to filter on).

2. You join customers_dataset using customer_id.

3. You join order_items_dataset using order_id (this gets us the product and the seller reference).

4. Finally, you join sellers_dataset using seller_id.

Since you nailed the underlying logic in Query 1, I expect your joins look solid. However, when dealing with 4 tables, aliasing is absolutely mandatory to keep your code readable and prevent errors.

In [ ]:
%%sql
--Here is the "Golden Standard" for how this query should look in the industry:
SELECT
    c.customer_id,
    c.customer_city,
    oi.product_id,
    s.seller_city
FROM orders_dataset o
JOIN customers_dataset c
    ON o.customer_id = c.customer_id
JOIN order_items_dataset oi
    ON o.order_id = oi.order_id
JOIN sellers_dataset s
    ON oi.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
LIMIT 10;

Running query in 'sqlite:///olist_ecommerce.db'

,customer_id,customer_city,product_id,seller_city
0,9ef432eb6251297304e76186b10a928d,sao paulo,87285b34884572647811a353c7ac498a,maua
1,b0830fb4747a6c6d20dea0b8c802d7ef,barreiras,595fac2a385ac33a80bd5114aec74eb8,belo horizonte
2,41ce2a54c0b03bf3443c3d931a367089,vianopolis,aa4383b373c6aca5d8797843e5594415,guariba
3,f88197465ea7920adcdbec7375364d82,sao goncalo do amarante,d0b61bfb1de832b15ba9d266ca96e5b0,belo horizonte
4,8ab97904e6daea8866dbdbc4fb7aad2c,santo andre,65266b2da20d04dbe00c5c2d3bb7859e,mogi das cruzes
5,503740e9ca751ccdda7ba28e9ab8f608,congonhinhas,060cb19345d90064d1015407193c233d,guarulhos
6,9bdf08b4b3b52b5526ff42d37d47f222,nilopolis,4520766ec412348b8d4caa5e8a18c464,atibaia
7,f54a9f0e6b351c431402b8461ea51999,faxinalzinho,ac1789e492dcd698c5c10b97a671243a,sao jose do rio pardo
8,31ad1d1b63eb9962463f764d4e6e0c9d,sorocaba,9a78fb9862b10749a117f7fc3c31f051,itaquaquecetuba
9,494dded5b201313c64ed7f100595b95c,rio de janeiro,08574b074924071f4e201e151b152b4e,cariacica


## **Master Query 3: The "Ghosted" Orders (LEFT JOIN & IS NULL)**

Concepts Tested: LEFT JOIN, WHERE ... IS NULL. (This replaces 20+ questions in your PDFs that ask "Find customers who never bought anything" or "Find employees without a department").

The Business Question: "Customer service wants to know which orders were placed but NEVER received a customer review. Find all order_ids that exist in the orders table but do not have a corresponding record in the reviews table."

Tables Needed: orders_dataset, order_reviews_dataset.

In [ ]:
%%sql
SELECT ORDERS_DATASET.ORDER_ID
FROM  ORDERS_DATASET
LEFT JOIN ORDER_REVIEWS_DATASET
ON ORDERS_DATASET.ORDER_ID = ORDER_REVIEWS_DATASET.ORDER_ID
WHERE ORDER_REVIEWS_DATASET.ORDER_ID IS NULL
;

Running query in 'sqlite:///olist_ecommerce.db'

,order_id
0,403b97836b0c04a622354cf531062e5f
1,6942b8da583c2f9957e990d028607019
2,4906eeadde5f70b308c20c4a8f20be02
3,b7a4a9ecb1cd3ef6a3e36a48e200e3be
4,59b32faedc12322c672e95ec3716d614
...,...
763,0c384d67524b5b92aa2fa6c8baa9a983
764,906a6b0a96d89ee226e4977e99b80b9e
765,5333db16fe357175d39c82840dd3269d
766,2f2df159f26ddb73d55ee72372200d3e


Let's keep the momentum going and move to Master Query 3: The "Ghosted" Orders.

Master Query 3: The "Ghosted" Orders
This question is testing a very specific, highly-tested industry pattern known as the "Anti-Join" or the "Orphan Hunt." When interviewers ask questions like "Find customers who never bought anything" or "Find items that were never returned," they are testing to see if you know how to combine a LEFT JOIN with a WHERE ... IS NULL clause.

The Goal: Find orders that were placed but NEVER reviewed.

The Logic & Review:
If you use an INNER JOIN here, you only get orders that do have reviews. That's the opposite of what we want.
Instead, you must use a LEFT JOIN to grab ALL orders, attach the reviews where they exist, and leave blanks (NULL) where they don't. Then, you filter for those blanks!

Looking directly at your work for Master Query 3: You nailed the logic perfectly. You grabbed the LEFT JOIN and successfully filtered the orphans using the IS NULL check. Your output is exactly what it should be.

## **Master Query 4: The Heavy Hitters (HAVING Clause)**

Concepts Tested: GROUP BY, HAVING, Aggregations.

The Business Question: "Which product categories are actually driving high volume? Return the product_category_name and the total number of products in that category, but only include categories that have more than 1,000 products."

Tables Needed: products_dataset.

In [ ]:
%%sql
SELECT product_category_name, COUNT(product_id) AS total_products
FROM PRODUCTS_DATASET
GROUP BY product_category_name
HAVING total_products > 1000
ORDER BY total_products DESC;

Running query in 'sqlite:///olist_ecommerce.db'

,product_category_name,total_products
0,cama_mesa_banho,3029
1,esporte_lazer,2867
2,moveis_decoracao,2657
3,beleza_saude,2444
4,utilidades_domesticas,2335
5,automotivo,1900
6,informatica_acessorios,1639
7,brinquedos,1411
8,relogios_presentes,1329
9,telefonia,1134


## **Master Query 5: The Whale Hunter (Scalar Subquery)**

Concepts Tested: Subqueries using =, MAX().

The Business Question: "Who made the single largest payment in the history of our platform? Write a query using a subquery to find the order_id and payment_value where the payment value is equal to the absolute maximum payment value in the database."

Tables Needed: order_payments_dataset.

In [ ]:
%%sql
SELECT order_id, payment_value
FROM order_payments_dataset
WHERE payment_value = (
  SELECT MAX(payment_value)
  FROM order_payments_dataset
  );

Running query in 'sqlite:///olist_ecommerce.db'

,order_id,payment_value
0,03caa2c082116e1d31e67e9ae3700499,13664.08


## **Master Query 6: The "Above Average" Sellers (Subquery with Aggregation)**

Concepts Tested: Advanced Subqueries, AVG().

The Business Question: "We want to find our premium items. List the product_id and price of all items that are priced strictly higher than the overall average price of all items in the dataset. Order them from most expensive to least."

Tables Needed: order_items_dataset.

In [ ]:
%%sql
SELECT PRODUCT_ID, price
FROM  order_items_dataset
WHERE PRICE > (SELECT AVG(PRICE) FROM  order_items_dataset)
ORDER BY PRICE DESC;

Running query in 'sqlite:///olist_ecommerce.db'

,product_id,price
0,489ae2aa008f021502940f251d4cce7f,6735.0
1,69c590f7ffc7bf8db97190b6cb6ed62e,6729.0
2,1bdf5e6731585cf01aa8169c7028d6ad,6499.0
3,a6492cc69376c469ab6f61d8f44de961,4799.0
4,c3ed642d592594bb648ff4a04cee2747,4690.0
...,...,...
32123,ba9f160a6ae1c23f25d690fd06fe4fd8,120.9
32124,fbec390384fbb8d53d97c29a5bbb26ae,120.9
32125,fbec390384fbb8d53d97c29a5bbb26ae,120.9
32126,ba9f160a6ae1c23f25d690fd06fe4fd8,120.9


## **Master Query 7: Monthly Revenue Trends (Dates + Joins + Grouping)**

Concepts Tested: Date formatting (e.g., STRFTIME() in SQLite or DATE_TRUNC() in Postgres), JOIN, GROUP BY, SUM().

The Business Question: "Leadership wants a high-level revenue chart. Calculate the total revenue (sum of payment_value) grouped by the Year and Month of the order_purchase_timestamp. Only include orders where the status is 'delivered'."

Tables Needed: orders_dataset, order_payments_dataset.

https://docs.google.com/document/d/1938K71cO5YVA_dP5Nzfry7nJwzLUNP01RFAhAE1Ku2g/edit?tab=t.0

In [ ]:
%%sql
SELECT
    STRFTIME('%Y-%m', o.order_purchase_timestamp) AS order_month,
    SUM(op.payment_value) AS total_revenue
FROM orders_dataset o
JOIN order_payments_dataset op
    ON o.order_id = op.order_id
WHERE o.order_status = 'delivered'
GROUP BY order_month
ORDER BY order_month ASC;

Running query in 'sqlite:///olist_ecommerce.db'

,order_month,total_revenue
0,2016-10,46566.71
1,2016-12,19.62
2,2017-01,127545.67
3,2017-02,271298.65
4,2017-03,414369.39
5,2017-04,390952.18
6,2017-05,567066.73
7,2017-06,490225.60
8,2017-07,566403.93
9,2017-08,646000.61



#Tier 3 Master Checklist

## Part 1: The Missing Tier 2 Concepts (Data Cleaning & Categorization)

1. String Manipulation: UPPER(), LOWER(), SUBSTR(), REPLACE() (Essential for cleaning messy real-world data).

2. Set Operations: UNION and UNION ALL (Stacking tables on top of each other instead of joining them side-by-side).

3. Conditional Logic: CASE WHEN ... THEN ... ELSE ... END (Creating custom buckets, flags, and pivoting data).

## Part 2: The Core Tier 3 Concepts (Advanced Analytics)

1. Common Table Expressions (CTEs): The WITH clause (Breaking down massive, complex queries into readable, step-by-step temporary tables).

2. Ranking Window Functions: ROW_NUMBER(), RANK(), DENSE_RANK() (Finding the top N items per category, not just overall).

3. Positional Window Functions: LAG() and LEAD() (Time-traveling within your data to compare a row with the previous or next row—crucial for Month-over-Month growth).

4. Window Aggregations: SUM() OVER() and AVG() OVER() (Calculating running totals and moving averages without collapsing the rows like a normal GROUP BY would).

5. Advanced Subqueries: Correlated subqueries and EXISTS / NOT EXISTS logic.

## Part 1: The Missing Tier 2 Concepts (Data Cleaning & Categorization)

### Master Query 1: The Unified Operations Report (Data Prep)


Concepts Tested: WITH (CTE), UNION ALL, UPPER(), SUBSTR() (String Manipulation).


The Business Question: Operations wants a single, unified report looking at orders, but they want it heavily customized. Write a query that grabs both delivered orders and canceled orders and stacks them together in the same list. For shipping labels, clean up the customer city names by converting them to uppercase and extracting just the first 3 letters to create a shortened "City Code". Wrap this entire foundational query inside a CTE so it acts as a clean, combined dataset.

Tables Needed: orders_dataset, customers_dataset.

**The Logic (What we are trying to do)**

Imagine you have two separate piles of paperwork on your desk: one pile for "Delivered" orders and one for "Canceled" orders.

Your boss wants you to combine them into one single, neat stack.

While you are stacking them, your boss also wants you to fix the shipping labels: instead of writing the full city name like "sao paulo", they just want a capitalized 3-letter code: "SAO".

Finally, they want you to put this entire clean stack into a neat folder so anyone else can easily pull reports from it later.

The Functions (Your Toolkit)

1. String Manipulation: SUBSTR() and UPPER()
Sometimes data is messy, and you need to slice it up.

    * SUBSTR(column_name, starting_position, number_of_characters) acts like a pair of scissors. If the city is "sao paulo", SUBSTR(customer_city, 1, 3) tells SQL: "Start at the 1st letter, and cut out exactly 3 letters." This gives us "sao".

    * UPPER() acts like a caps-lock key. If we wrap our scissors inside it like this: UPPER(SUBSTR(customer_city, 1, 3)), it turns "sao" into "SAO".

2. Stacking Data: UNION ALL
    * You already know that JOIN puts tables side-by-side (like adding new columns). UNION ALL puts tables top-to-bottom (like stacking rows on top of each other).

    * We will write one query for Delivered orders, put UNION ALL in the middle, and write a second query for Canceled orders. SQL will glue them into one long vertical list.

3. The CTE: WITH clause
    * A Common Table Expression (CTE) is just a temporary, invisible box where you store your messy work. It makes your final code look incredibly clean. You just write WITH NameOfYourBox AS ( ...your messy query... ) and then you can simply run SELECT * FROM NameOfYourBox; at the very end.

In [ ]:
%%sql
-- the code
-- Step 1: We create our temporary "folder" (CTE) called UnifiedOrders
WITH UnifiedOrders AS (

    -- Step 2: Grab the Delivered Orders and clean the city name
    SELECT
        o.order_id,
        o.order_status,
        UPPER(SUBSTR(c.customer_city, 1, 3)) AS city_code
    FROM orders_dataset o
    JOIN customers_dataset c
        ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'

    -- Step 3: Stack the next query directly underneath using UNION ALL
    UNION ALL

    -- Step 4: Grab the Canceled Orders and clean the city name
    SELECT
        o.order_id,
        o.order_status,
        UPPER(SUBSTR(c.customer_city, 1, 3)) AS city_code
    FROM orders_dataset o
    JOIN customers_dataset c
        ON o.customer_id = c.customer_id
    WHERE o.order_status = 'canceled'

)

-- Step 5: Now, the actual query is incredibly simple. We just query our CTE!
SELECT *
FROM UnifiedOrders;


Running query in 'sqlite:///olist_ecommerce.db'

,order_id,order_status,city_code
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,SAO
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,BAR
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,VIA
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,SAO
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,SAN
...,...,...,...
97098,b159d0ce7cd881052da94fa165617b05,canceled,GAS
97099,e49e7ce1471b4693482d40c2bd3ad196,canceled,SAO
97100,6560fb10610771449cb0463c5ba12199,canceled,CUR
97101,3a3cddda5a7c27851bd96c3313412840,canceled,SAN


### Master Query 2: The Custom Categorizer


Concepts Tested: CASE WHEN, Advanced Grouping, WITH (CTE).

The Business Question: We need to evaluate our revenue streams. Use conditional logic to label individual order items as "High Value", "Medium Value", or "Low Value" based on their price. Wrap this categorization inside a CTE, and then query that CTE to calculate exactly what percentage of our total platform revenue comes specifically from our "High Value" bucket.

Tables Needed: order_items_dataset.

The Logic (What we are trying to do)

Imagine you are looking at a massive spreadsheet of every single item ever sold in your store. You have thousands of prices ranging from USD 5 to USD 5,000.

Your boss doesn't want to see individual prices. They want you to go down the list with three colored stickers: Gold ("High Value"), Silver ("Medium Value"), and Bronze ("Low Value") and slap a sticker on every single row based on the price.

Once everything has a sticker, you put that whole list into a temporary box.

Finally, you want to ask one simple math question: "Out of all the money we made, what exact percentage came from the items with the Gold ('High Value') stickers?"

The Functions (Your Toolkit)

1. The Sorting Hat: CASE WHEN
This is SQL's way of doing an "IF / THEN" statement. You are giving the database a set of rules to follow row by row.

   * Rule 1: WHEN the price is greater than $1,000, THEN call it 'High Value'.

   * Rule 2: WHEN the price is greater than $100, THEN call it 'Medium Value'.

   * Catch-all: ELSE (if it doesn't fit the above), call it 'Low Value'.

   * Every CASE statement must finish with the word END.

2. The Temporary Box: WITH (CTE)
   * Just like in Query 1, we will use a Common Table Expression (CTE) to hold our "stickered" data. We do this so that in the next step, calculating the math is super easy.

3. Conditional Math: SUM(CASE WHEN...)
   * To find a percentage, you need two numbers: the Part (High Value Revenue) divided by the Whole (Total Revenue).

   * To get the "Whole", we just use normal SUM(price).

   * To get the "Part", we put a CASE WHEN inside a SUM(). We tell SQL: "If the sticker is 'High Value', add the price to my total. If it's not, add 0."

In [ ]:
%%sql
-- Step 1: Create our temporary box (CTE) and label every single item
WITH CategorizedItems AS (
    SELECT
        order_id,
        price,
        -- Here is our "Sorting Hat" logic
        CASE
            WHEN price >= 1000 THEN 'High Value'
            WHEN price >= 100 THEN 'Medium Value'
            ELSE 'Low Value'
        END AS value_bucket
    FROM order_items_dataset
)

-- Step 2: Now we look inside our box and do the math
SELECT
    -- 1. Get the total money from ALL items
    SUM(price) AS total_revenue,

    -- 2. Get the money ONLY from High Value items
    SUM(CASE WHEN value_bucket = 'High Value' THEN price ELSE 0 END) AS high_value_revenue,

    -- 3. Divide the High Value revenue by the Total revenue and multiply by 100 to get a %
    (SUM(CASE WHEN value_bucket = 'High Value' THEN price ELSE 0 END) / SUM(price)) * 100 AS high_value_percentage

FROM CategorizedItems;

Running query in 'sqlite:///olist_ecommerce.db'

,total_revenue,high_value_revenue,high_value_percentage
0,1.359164e+07,1343114.12,9.881911


Let's strip away the OUTPUT and translate that screenshot into plain English.

**What are we doing here? (The Business Goal)**

Imagine your boss walks up to you and says: *"Hey, we sell a lot of cheap stuff and a few really expensive items (over USD 1,000). I want to know exactly how much our overall business relies on those super expensive items.

That one row of output in your screenshot answers that exact question perfectly. Let's look at the three columns you generated:

1. total_revenue (13,594,107.12...): This is the entire pie. If you add up the price of every single item ever sold on your platform, the company made roughly 13.59 million

2. high_value_revenue (1,386,762.30...): This is the golden slice of the pie. This is the total money made only from the items that cost usd 1,000 or more. You've made about $1.38 million from just those expensive items.

3. high_value_percentage (10.2012...): This is the punchline your boss asked for. The database took the golden slice (1.38 million) and divided it by the whole pie (13.59 million). It tells us that 10.2% of your entire company's revenue comes from those high-value items.


## Part 2: The Core Tier 3 Concepts (Advanced Analytics)

### Master Query 3: The Leaderboard


Concepts Tested: RANK(), DENSE_RANK(), ROW_NUMBER(), PARTITION BY.


The Business Question: Finding the single most expensive product on the platform is easy. But we need to find the #1 most expensive product in every single product category. Use a Window Function to partition the dataset by category and rank the items by price descending. Filter your final results to only show the products that hold the #1 rank in their respective groups.

Tables Needed: products_dataset, order_items_dataset

The Logic (What we are trying to do)
1. Imagine you are the principal of a high school, and you have a giant list of every student and their test scores.

2. If you want to find the single smartest student in the whole school, that is easy. You just sort by score and look at the top name.

3. But what if you want to find the #1 smartest student in EVERY single classroom?

4. You would need to go room by room, hand out a gold medal to the top score in that room, and then reset your medals before walking into the next room.

**That is what we are doing here. We want to find the #1 most expensive product, but we want to do it for every single product category (Electronics, Furniture, Toys, etc.)**

The Functions (Your Toolkit)

1. The Window Function: DENSE_RANK() OVER (...)
     * Normally, if you use a GROUP BY, SQL squashes all your rows together and you lose the details (like the product ID). A Window Function is magical because it lets you calculate things (like rankings) without squashing the rows. It just adds a new column with the rank right next to the original data.

     * DENSE_RANK() acts like a judge handing out medals (1st place, 2nd place, 3rd place).

2. The Reset Button: PARTITION BY
     * Inside your Window Function, you have to tell SQL how to hand out the medals.

     * PARTITION BY product_category tells SQL: "Every time you look at a new category, take back your medals and start counting from #1 again." (This is how we go "room by room").

     * ORDER BY price DESC tells SQL who gets the #1 medal (the highest price gets #1).

3. The Temporary Box: WITH (CTE)
     * Why do we need a CTE again? Because SQL is a little rigid. It calculates your rankings at the very end of its process. You are not allowed to say WHERE rank = 1 in the exact same query where you create the rank.

So, we rank everyone, put that giant list into our temporary box (CTE), and then in the next step, we just tell the box: "Only let out the products holding a #1 medal."

In [ ]:
%%sql
-- Step 1: Create our temporary box (CTE) and hand out all the medals
WITH RankedProducts AS (
    SELECT
        p.product_category_name,
        p.product_id,
        oi.price,

        -- Here is the Window Function magic!
        DENSE_RANK() OVER (
            PARTITION BY p.product_category_name -- "Reset the medals for each category"
            ORDER BY oi.price DESC               -- "#1 goes to the highest price"
        ) AS category_rank

    FROM products_dataset p
    JOIN order_items_dataset oi
        ON p.product_id = oi.product_id
    WHERE p.product_category_name IS NOT NULL
)

-- Step 2: Now we look inside our box, and ONLY grab the #1 winners
SELECT
    product_category_name,
    price
FROM RankedProducts
WHERE category_rank = 1;

Running query in 'sqlite:///olist_ecommerce.db'

,product_category_name,price
0,agro_industria_e_comercio,2990.00
1,alimentos,274.99
2,alimentos_bebidas,699.90
3,artes,6499.00
4,artes_e_artesanato,289.49
...,...,...
92,telefonia_fixa,1790.00
93,telefonia_fixa,1790.00
94,telefonia_fixa,1790.00
95,telefonia_fixa,1790.00
